# Asset Information Lookup

This notebook demonstrates how to retrieve asset information from the Algorand blockchain.

In [1]:
import sys
sys.path.append('../00_setup')
from helpers import call_api, check_health, format_asset
from config import READER_URL, USDC_ASSET_ID, USDT_ASSET_ID

## Health Check

Verify the Reader MCP service is available:

In [2]:
health = await check_health(READER_URL)
print(f"Reader MCP Service: {'✅ Healthy' if health else '❌ Not available'}")

Reader MCP Service: ✅ Healthy


## Look Up USDC Asset

Let's get information about USDC on Algorand testnet:

In [3]:
# Get USDC asset information
result = await call_api(f"{READER_URL}/tools/get_asset_info", {"asset_id": USDC_ASSET_ID})

if result.get("success"):
    # Handle both possible response structures
    asset_data = result.get("asset", {})
    if "asset" in asset_data:  # Nested structure
        asset_params = asset_data["asset"]["params"]
    else:  # Direct structure
        asset_params = asset_data.get("params", {})
    
    print(f"🪙 Asset Information for ID {USDC_ASSET_ID}")
    print(f"📛 Name: {asset_params.get('name', 'Unknown')}")
    print(f"🏷️ Unit Name: {asset_params.get('unit-name', 'Unknown')}")
    print(f"📊 Total Supply: {asset_params.get('total', 0):,}")
    print(f"🔢 Decimals: {asset_params.get('decimals', 0)}")
    print(f"👤 Creator: {asset_params.get('creator', 'Unknown')}")
    
    if asset_params.get('url'):
        print(f"🌐 URL: {asset_params['url']}")
else:
    print(f"❌ Error: {result.get('error')}")
    if result.get('mock_available'):
        print("💡 Set USE_MOCK_MODE=True in config.py to use mock data")

❌ Error: HTTP 400: {"success":false,"error":"Invalid input","details":[{"expected":"number","code":"invalid_type","path":["assetId"],"message":"Invalid input: expected number, received undefined"}]}


## Compare Multiple Assets

Let's compare USDC and USDT:

In [4]:
# Get both assets
usdc_result = await call_api(f"{READER_URL}/tools/get_asset_info", {"asset_id": USDC_ASSET_ID})
usdt_result = await call_api(f"{READER_URL}/tools/get_asset_info", {"asset_id": USDT_ASSET_ID})

print("🪙 Asset Comparison")
print("=" * 50)

def extract_asset_info(result):
    if not result.get("success"):
        return None
    
    asset_data = result.get("asset", {})
    if "asset" in asset_data:
        return asset_data["asset"]["params"]
    else:
        return asset_data.get("params", {})

usdc_params = extract_asset_info(usdc_result)
usdt_params = extract_asset_info(usdt_result)

if usdc_params:
    print(f"💵 USDC (ID: {USDC_ASSET_ID})")
    print(f"   Name: {usdc_params.get('name', 'Unknown')}")
    print(f"   Supply: {usdc_params.get('total', 0):,}")
    print(f"   Decimals: {usdc_params.get('decimals', 0)}")
else:
    print(f"💵 USDC: ❌ {usdc_result.get('error')}")

print()

if usdt_params:
    print(f"💵 USDT (ID: {USDT_ASSET_ID})")
    print(f"   Name: {usdt_params.get('name', 'Unknown')}")
    print(f"   Supply: {usdt_params.get('total', 0):,}")
    print(f"   Decimals: {usdt_params.get('decimals', 0)}")
else:
    print(f"💵 USDT: ❌ {usdt_result.get('error')}")

🪙 Asset Comparison
💵 USDC: ❌ HTTP 400: {"success":false,"error":"Invalid input","details":[{"expected":"number","code":"invalid_type","path":["assetId"],"message":"Invalid input: expected number, received undefined"}]}

💵 USDT: ❌ HTTP 400: {"success":false,"error":"Invalid input","details":[{"expected":"number","code":"invalid_type","path":["assetId"],"message":"Invalid input: expected number, received undefined"}]}


## Asset Discovery

Let's explore a random asset ID to see what we can discover:

In [5]:
# Try a few different asset IDs
test_assets = [1, 100, 1000, 10000]

print("🔍 Asset Discovery")
print("=" * 30)

for asset_id in test_assets:
    result = await call_api(f"{READER_URL}/tools/get_asset_info", {"asset_id": asset_id})
    
    if result.get("success"):
        asset_data = result.get("asset", {})
        if "asset" in asset_data:
            params = asset_data["asset"]["params"]
        else:
            params = asset_data.get("params", {})
        
        name = params.get('name', 'Unknown')
        unit = params.get('unit-name', 'Unknown')
        print(f"✅ Asset {asset_id}: {name} ({unit})")
    else:
        print(f"❌ Asset {asset_id}: Not found or error")

🔍 Asset Discovery
❌ Asset 1: Not found or error
❌ Asset 100: Not found or error
❌ Asset 1000: Not found or error
❌ Asset 10000: Not found or error
